# Knowledge Context API Design

**Date:** 2026-06-07

**Design epic:** `bd-4rh7`

**Status:** draft spec for review

## Objective

Design a one-shot MCP context-pack API that composes SPUR's existing knowledge layers into bounded, evidence-first context bundles for agents. The API should reduce manual tool chaining while preserving the distinction between approximate retrieval and exact source-of-truth graph operations.

This spec is intentionally implementation-oriented, but it does not authorize code changes by itself. It records the proposed architecture, boundaries, phased rollout, and verification strategy.

## Current Implementation Findings

The current knowledge layer is strong but fragmented across separate surfaces:

| Layer | Current owner | Current role |
|---|---|---|
| Structural graph artifact | `crates/spur-graph` | Builds symbols, files, edges, temporal facts, and Parquet artifact directories. |
| LanceDB section sidecar | `crates/spur-graph/src/store/lance_sections.rs` | Stores markdown/prose section bodies, optional embedding vectors, FTS/vector indexes. |
| Analyst DuckDB | `crates/spur-cli/src/commands/analyst.rs` + `crates/spur-context/poc/duckdb-analyst/*.sql` | Builds `.spur/analyst.duckdb`; materializes BM25 corpora, graph scorecards, PageRank, churn, posture, graph macros. |
| Exact code graph MCP | `crates/spur-mcp` | Exposes exact symbol lookup, source reads, callers, callees, and subgraph tools. |
| Semantic search MCP | `crates/spur-notebook/src/mcp/tools/code_semantic_search.rs` | Opens analyst DuckDB read-only and runs `docs`, `code`, `all`, `graph`, or optional `hybrid` search. |

Live local analyst DB sampled during design:

- nodes: `52,356`
- resolved edges: `109,366` in `_meta`, `109,470` through the re-keyed `edges` view
- unresolved edges: about `95k`
- searchable document sections: `18,107`
- searchable code symbol rows: `32,921`
- temporal commits: `3,033`
- symbol snapshots: `610,929`

Important boundary: Lance ANN is a quality enhancer for fuzzy prose retrieval. It is not the foundation of the knowledge layer. BM25, scorecards, exact symbol graph, and call graph expansion all work without ANN.

## Current Architecture

```mermaid
flowchart TB
  subgraph Build[Build-Time Knowledge Materialization]
    Source[Workspace files + git history]
    GraphBuilder[spur-graph builder]
    Artifact[Graph artifact directory\nParquet: nodes, edges, files, temporal shards]
    LanceSidecar[sections.lancedb\nsection_bodies table\nbody_text FTS + optional vector index]
    AnalystBuild[spur-cli analyst build\nDuckDB CLI orchestration]
    AnalystDB[(.spur/analyst.duckdb)]
  end

  subgraph Query[Query-Time Retrieval Surfaces]
    NotebookSearch[notebook MCP\ncode_semantic_search]
    GraphMCP[spur-mcp code_* tools\nexact symbol graph]
    HumanAgent[Agent / user]
  end

  Source --> GraphBuilder
  GraphBuilder --> Artifact
  GraphBuilder --> LanceSidecar
  Artifact --> AnalystBuild
  LanceSidecar -->|ATTACH ... TYPE LANCE| AnalystBuild
  AnalystBuild --> AnalystDB
  AnalystDB -->|BM25 + scorecards + graph macros| NotebookSearch
  LanceSidecar -->|optional ANN for hybrid mode| NotebookSearch
  Artifact -->|exact graph index| GraphMCP
  HumanAgent --> NotebookSearch
  HumanAgent --> GraphMCP

  classDef durable fill:#eef7ff,stroke:#3b82f6,color:#111827;
  classDef query fill:#f7fee7,stroke:#65a30d,color:#111827;
  class Artifact,LanceSidecar,AnalystDB durable;
  class NotebookSearch,GraphMCP query;
```

The architectural gap is not missing data. The gap is that context assembly is manual: an agent must search semantically, inspect exact symbols, ask for callers/callees, inspect docs, and reason about scorecards as separate steps.

## First-Principles Design Review

A useful agent knowledge layer must satisfy four properties:

1. **Recall:** find relevant areas when the user's wording does not match exact identifiers.
2. **Grounding:** attach exact source, file, line, symbol IDs, and graph hashes before claims are made.
3. **Impact:** show what depends on the candidate, what it calls, and whether it is load-bearing.
4. **Budget control:** return a bounded evidence pack instead of flooding the context window.

From those properties, the role split is:

| Capability | Best current substrate | Why |
|---|---|---|
| Broad concept discovery | `code_semantic_search` / analyst BM25 / optional ANN | High recall over docs and symbol token text. |
| Exact source truth | `code_read_symbol` / graph artifact | Stable symbol IDs and exact source slices. |
| Change impact | `code_callers`, `code_callees`, analyst scorecards | Direct dependency surface plus centrality/churn/posture. |
| Cross-document rationale | `sections_search`, specs/plans/docs in analyst DB | Captures design docs and skill bodies. |
| Fuzzy prose matching | Lance ANN, optional | Helpful for wording mismatch; expensive dependency stack. |

Conclusion: semantic search should remain a candidate finder, not the source of authority. The new API should orchestrate retrieval and grounding.

## Proposed MCP Tool: `knowledge_context_pack`

### Purpose

Return a bounded, machine-readable evidence bundle for a natural-language engineering question.

### Request

```json
{
  "query": "can we remove lancedb from spur-graph?",
  "intent": "explain",
  "scope": "all",
  "limit": 8,
  "include_source": "summaries",
  "include_tests": true,
  "max_symbol_bodies": 3
}
```

### Intent Values

| Intent | Retrieval policy |
|---|---|
| `explain` | Prefer docs, current implementation symbols, dependency evidence. |
| `change` | Prefer code hits, callers/callees, tests, affected crates, risk score. |
| `review` | Prefer changed files, invariants, tests, docs, nearby high-risk symbols. |
| `debug` | Prefer error text hits, call path, tests, RCA docs, recent churn. |
| `plan` | Prefer specs/plans, ownership boundaries, components, dependency DAG hints. |

### Response Shape

```json
{
  "query": "can we remove lancedb from spur-graph?",
  "intent": "explain",
  "answerable": true,
  "confidence": "medium",
  "graph_content_hash": "1fd41da1...",
  "staleness": { "analyst_matches_graph": true },
  "summary": "LanceDB is used directly for section sidecar writes and reads; removing it requires replacing table/index/query APIs.",
  "primary_evidence": [],
  "supporting_docs": [],
  "impact": {},
  "recommended_next_tools": []
}
```

The tool should always preserve enough IDs and file references for exact follow-up calls. It should never hide that semantic matching is approximate.

## One-Shot Context Assembly Sequence

```mermaid
sequenceDiagram
  autonumber
  participant A as Agent
  participant K as knowledge_context_pack
  participant D as Analyst DuckDB
  participant L as Lance ANN optional
  participant G as Exact Graph Backend
  participant T as Test/Doc Resolver

  A->>K: query + intent + scope + budget
  K->>D: search/search_code/search_graph/search_docs_bm25
  D-->>K: ranked candidates + BM25 + scorecard signals

  alt scope == hybrid and ANN available
    K->>L: embed query + nearest_to(vector)
    L-->>K: ANN section IDs + distances
    K->>K: RRF fuse BM25 + ANN
  else ANN unavailable or disabled
    K->>K: BM25-only candidate set
  end

  K->>K: dedupe by stable_symbol_id/file and enforce per-doc cap
  K->>G: resolve top code candidates to symbol metadata
  G-->>K: stable IDs, file ranges, exact source summaries
  K->>G: callers/callees counts for top change-relevant symbols
  G-->>K: impact counts + direct neighbor summaries
  K->>T: find tests/specs/plans touching top files/symbols
  T-->>K: supporting evidence references
  K->>K: rank evidence by intent and token budget
  K-->>A: bounded evidence pack + recommended exact follow-ups
```

The one-shot call is not a replacement for exact graph tools. It is a routing and packing layer that performs the first high-signal retrieval pass.

## Component Boundaries

```mermaid
flowchart LR
  subgraph API[New API Surface]
    Tool[spur-mcp tool\nknowledge_context_pack]
    Policy[Intent policy + budgeter]
    Pack[Evidence pack serializer]
  end

  subgraph Existing[Existing Retrieval Primitives]
    Semantic[Analyst semantic SQL\nsearch/search_code/search_graph]
    Exact[code_* graph handlers\nread_symbol/callers/callees]
    Sidecar[Lance ANN optional\nsections.lancedb vectors]
    TestFinder[Test + spec resolver\nfile/symbol/doc matching]
  end

  subgraph Durable[Durable Data]
    Duck[(.spur/analyst.duckdb)]
    Parquet[(graph artifact parquet)]
    Lance[(sections.lancedb)]
  end

  Tool --> Policy
  Policy --> Semantic
  Policy --> Exact
  Policy -. optional .-> Sidecar
  Policy --> TestFinder
  Semantic --> Duck
  Exact --> Parquet
  Sidecar --> Lance
  TestFinder --> Duck
  Semantic --> Pack
  Exact --> Pack
  Sidecar --> Pack
  TestFinder --> Pack

  classDef new fill:#fef9c3,stroke:#ca8a04,color:#111827;
  classDef existing fill:#ecfdf5,stroke:#059669,color:#111827;
  classDef data fill:#eff6ff,stroke:#2563eb,color:#111827;
  class Tool,Policy,Pack new;
  class Semantic,Exact,Sidecar,TestFinder existing;
  class Duck,Parquet,Lance data;
```

### Placement Recommendation

Put the high-level tool in `spur-mcp`, backed by a shared internal module or crate if notebook also needs it later.

Reasoning:

- `spur-mcp` already owns worker-facing exact graph tools.
- Worker agents need this more than notebook-only users.
- Keeping the tool near `code_*` lets it reuse exact graph handlers without reimplementing symbol resolution.
- Notebook can later call the same MCP surface instead of maintaining a separate richer version.

Lance ANN should remain optional. A missing Lance sidecar or missing feature should degrade to BM25 + graph grounding.

## Evidence Pack Schema

### Top-Level Fields

| Field | Required | Description |
|---|---:|---|
| `query` | yes | Original natural-language request. |
| `intent` | yes | Retrieval policy selected by caller or inferred. |
| `answerable` | yes | Whether enough local evidence was found. |
| `confidence` | yes | `low`, `medium`, or `high`; based on evidence agreement and staleness. |
| `graph_content_hash` | yes | Hash from analyst DB and/or exact graph artifact. |
| `staleness` | yes | Whether analyst DB and graph artifact appear aligned. |
| `primary_evidence` | yes | Top grounded evidence items. |
| `supporting_docs` | no | Specs, plans, RCA docs, skills, or markdown sections. |
| `impact` | no | Caller/callee counts, affected crates, posture summary. |
| `recommended_next_tools` | yes | Exact follow-up calls for verification or deeper work. |

### Evidence Item Fields

```json
{
  "kind": "symbol|doc|test|dependency|sql_view",
  "title": "write_sections_dataset_async",
  "file": "crates/spur-graph/src/store/lance_sections.rs",
  "line_start": 378,
  "line_end": 547,
  "stable_symbol_id": "graph://symbol/1a81a27feb5d3e23",
  "score": 10.69,
  "signal": "load-bearing wall · pr=0.3 · churn=0",
  "why_relevant": "Connects to sections.lancedb and creates/opens the section table.",
  "source_summary": "Writes batched section rows into LanceDB and ensures FTS/vector indexes after dataset changes.",
  "grounding": "exact|semantic|ann|sql",
  "next": ["code_read_symbol", "code_callers"]
}
```

### Budget Policy

Default pack budget should be conservative:

- max primary evidence: `8`
- max symbol bodies summarized: `3`
- max docs: `5`
- max tests: `5`
- max caller/callee rows per symbol: counts plus top `3`, not full fan-out

Popular sinks should return counts and posture instead of expanding all neighbors.

## Intent Policy Matrix

```mermaid
flowchart TB
  Q[Incoming query] --> I{Intent}

  I -->|explain| E[Prefer docs + implementation symbols\nReturn concise evidence and dependency facts]
  I -->|change| C[Prefer code + callers/callees\nReturn impact, affected crates, tests]
  I -->|review| R[Prefer changed paths + invariants\nReturn risks, missing tests, docs impact]
  I -->|debug| D[Prefer error text + recent churn\nReturn call path, tests, RCA docs]
  I -->|plan| P[Prefer specs/plans + ownership boundaries\nReturn task slices and dependency hints]

  E --> BM25[Analyst BM25]
  C --> Graph[Exact graph expansion]
  R --> Score[Scorecard + changed files]
  D --> Temporal[Temporal/churn views]
  P --> Docs[Specs/plans/doc tree]

  BM25 --> Pack[Evidence pack]
  Graph --> Pack
  Score --> Pack
  Temporal --> Pack
  Docs --> Pack

  Pack --> Follow[Recommended exact follow-ups]
```

The tool can accept explicit intent first. Intent inference can be added later, but should not block the MVP.

## Phased Rollout

```mermaid
gantt
  title Knowledge Context API rollout
  dateFormat  YYYY-MM-DD
  axisFormat  %m-%d

  section Phase 0 - Spec
  Design notebook and review              :done, p0a, 2026-06-07, 1d

  section Phase 1 - BM25 + Exact Grounding
  Add tool schema and request parser       :p1a, after p0a, 1d
  Query analyst DB search macros           :p1b, after p1a, 1d
  Resolve top code hits to graph symbols   :p1c, after p1b, 2d
  Return bounded evidence pack             :p1d, after p1c, 1d

  section Phase 2 - Impact Context
  Add caller/callee counts                 :p2a, after p1d, 1d
  Add tests/specs resolver                 :p2b, after p1d, 2d
  Add staleness/hash checks                :p2c, after p1d, 1d

  section Phase 3 - Optional Hybrid
  Wire optional Lance ANN fallback         :p3a, after p2b, 2d
  Add feature-gated compile path           :p3b, after p3a, 2d

  section Phase 4 - Adoption
  Notebook calls shared tool               :p4a, after p3b, 2d
  Worker prompt guidance update            :p4b, after p3b, 1d
```

### MVP Cut

The MVP should skip ANN entirely and prove value with:

1. analyst BM25 candidates
2. exact graph grounding
3. caller/callee counts
4. scorecard signal
5. bounded JSON response

This avoids coupling the first delivery to the Lance/DataFusion compile-cost problem.

## Risks And Mitigations

| Risk | Impact | Mitigation |
|---|---|---|
| Semantic hit treated as truth | Incorrect engineering conclusions | Every evidence item carries `grounding`; exact graph/file references required for primary code evidence. |
| Context packs become too large | Token waste and slower agents | Hard caps by evidence type; counts before rows for popular symbols. |
| Analyst DB stale relative to graph artifact | Mismatched source evidence | Compare `graph_content_hash`; expose staleness in response. |
| Lance ANN compile/runtime dependency worsens iteration | Slower development and larger builds | Keep ANN optional and degrade to BM25 + graph. |
| Duplicated logic between notebook and `spur-mcp` | Divergent behavior | Put orchestration in `spur-mcp` or shared crate; notebook becomes consumer. |
| DuckDB CLI/extension availability blocks build | Missing analyst DB | Tool should report `analyst_unavailable` and fall back to exact graph when possible. |
| Popular-sink expansion floods results | Low-signal packs | Use caller count and posture gates from existing `search_graph` design. |

## Open Questions

1. Should `knowledge_context_pack` live only in `spur-mcp`, or should the core orchestration live in a small shared crate consumed by both `spur-mcp` and `spur-notebook`?
2. Should intent be required in the first version, or should the tool infer intent from query verbs?
3. Should source bodies be included inline, summarized, or returned only as stable references by default?
4. What is the default fallback behavior when `.spur/analyst.duckdb` is missing but the graph artifact exists?
5. Should dependency evidence from `cargo tree` be in scope for the MVP, or left as a recommended follow-up command?

## Acceptance Criteria

The design is ready for implementation planning when:

- The tool contract is stable enough for worker agents.
- The MVP excludes optional ANN and still returns useful context packs.
- Exact graph grounding is mandatory for primary code evidence.
- Staleness metadata is present in every response.
- Popular-sink expansion is bounded.
- Tests can use fixture analyst DBs and graph artifacts without requiring a live full repo index.

## Recommendation

Build `knowledge_context_pack` in two layers:

1. **MVP:** `spur-mcp` tool using analyst BM25 + exact graph grounding + scorecards.
2. **Enhancement:** optional ANN and richer notebook integration after the base API proves stable.

This delivers the one-shot MCP call the agents need while keeping the expensive Lance/DataFusion path optional rather than more central.